# Chapter 5: Duality

## Overview
Duality is one of the most profound concepts in optimization. Every optimization problem (the **primal** problem) has an associated **dual** problem. 

### Key Concepts
1. **The Lagrangian:** We augment the objective function with a weighted sum of the constraint functions. For a problem with inequality constraints $f_i(x) \le 0$ and equality constraints $h_i(x) = 0$, the Lagrangian is:
   
   $$
   L(x, \lambda, \nu) = f_0(x) + \sum_{i=1}^m \lambda_i f_i(x) + \sum_{i=1}^p \nu_i h_i(x)
   $$
   
   where $\lambda \ge 0$ and $\nu$ are the **Lagrange multipliers** (or dual variables).

2. **The Lagrange Dual Function:** $g(\lambda, \nu) = \inf_x L(x, \lambda, \nu)$. 
   - $g$ is *always* concave, even if the original primal problem is not convex!
   - $g(\lambda, \nu)$ provides a guaranteed lower bound on the optimal primal value $p^\star$.

3. **Weak and Strong Duality:**
   - **Weak Duality:** The optimal dual value $d^\star$ is always $\le p^\star$. The difference $p^\star - d^\star$ is the "duality gap".
   - **Strong Duality:** $d^\star = p^\star$ (zero duality gap). This usually holds for convex problems that strictly satisfy their constraints (known as **Slater's Condition**).

4. **KKT Conditions:** The Karush-Kuhn-Tucker conditions are the necessary and sufficient conditions for optimality in a convex problem with differentiable functions and strong duality. They include:
   - Primal feasibility
   - Dual feasibility
   - Complementary slackness ($\lambda_i f_i(x) = 0$)
   - Gradient of the Lagrangian with respect to $x$ vanishes.

## Applications & Problems Solved
- **Certificate of Optimality:** Dual variables provide a way to *prove* you have found the optimal solution (by checking if the duality gap is zero).
- **Sensitivity Analysis (Shadow Prices):** The optimal dual variables $\lambda^\star$ indicate exactly how much the optimal objective value would improve if a constraint were relaxed by a small amount. In economics, these are "shadow prices".
- **Algorithm Design:** Many state-of-the-art solvers (like Primal-Dual Interior-Point methods) do not just solve for $x$, they simultaneously solve for $(x, \lambda, \nu)$ using the KKT conditions.

## Code Example
See `kkt_and_duality.py` to see how to:
1. Define and solve a primal linear program.
2. Formulate its theoretical dual problem and solve it.
3. Verify **Strong Duality** (the values are equal).
4. Extract the optimal dual variables (shadow prices) directly from a solver.


In [ ]:
%matplotlib inline
import cvxpy as cp
import numpy as np

def demonstrate_strong_duality():
    """
    Demonstrates Strong Duality by solving both the Primal
    and the Dual of a Linear Program.
    
    Primal (Standard Form LP):
    Minimize: c^T x
    Subject to: Ax = b, x >= 0
    
    Dual LP:
    Maximize: -b^T nu
    Subject to: -A^T nu + c >= 0
    """
    np.random.seed(42)
    m, n = 5, 10
    
    # Generate random data. Ensure A, b, c give a feasible bounded LP
    A = np.random.randn(m, n)
    # create a feasible x to ensure A x = b is possible for x >= 0
    x0 = np.random.rand(n) 
    b = A @ x0
    c = np.random.rand(n) # positive cost to ensure boundedness

    print("--- 1. Solving Primal Problem ---")
    x = cp.Variable(n)
    primal_constraints = [A @ x == b, x >= 0]
    primal_prob = cp.Problem(cp.Minimize(c.T @ x), primal_constraints)
    primal_prob.solve()
    
    p_star = primal_prob.value
    print(f"Primal Optimal Value (p*): {p_star:.4f}")

    print("\n--- 2. Solving Dual Problem ---")
    # For constraints Ax = b, dual variable is nu (free)
    # For constraints x >= 0 (-x <= 0), dual variable is lambda >= 0
    # Following standard LP duality derived from the Lagrangian
    nu = cp.Variable(m)
    dual_constraints = [-A.T @ nu + c >= 0]
    dual_prob = cp.Problem(cp.Maximize(-b.T @ nu), dual_constraints)
    dual_prob.solve()
    
    d_star = dual_prob.value
    print(f"Dual Optimal Value (d*): {d_star:.4f}")
    
    print("\n--- 3. Verifying Strong Duality ---")
    gap = abs(p_star - d_star)
    print(f"Duality Gap |p* - d*| = {gap:.4e}")
    if gap < 1e-5:
        print("Strong duality holds!")
        
    print("\n--- 4. Extracting Dual Variables (Shadow Prices) from Primal ---")
    # cvxpy automatically stores the dual variables in the constraints after solve
    nu_star_extracted = primal_constraints[0].dual_value
    print(f"Dual variable (nu) from primal constraint: {nu_star_extracted}")
    print(f"Dual variable (nu) from dual solver: {nu.value}")
    
if __name__ == "__main__":
    demonstrate_strong_duality()
